# Statistical Inference – Promo2 & Robot Arm
**Course:** Data & Analytics | Prof. Dr. Sonja Schneider  
**Institution:** Technische Hochschule Nürnberg Georg Simon Ohm  

**Contributors:**
- Danny Schönhals
- Athithya Mariyanayagam

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

# %matplotlib inline

# # Pre-processing tip: statsmodels drops NaNs automatically,
# # but it's cleaner to handle them explicitly before fitting, e.g.:
# df_clean = df.dropna(subset=['y', 'x1', 'x2'])

# # fit the model
# model = smf.ols("y ~ x1 + x2", data=df).fit()  # "y~x1+x2" is the formula for Ordinary Least Squares (OLS)
# print(model.summary())

# # get the values we're interested in
# params = model.params            # coefficients
# pvals  = model.pvalues           # p-values
# conf   = model.conf_int()        # confidence intervals
# r2     = model.rsquared          # R2
# res    = model.resid             # residuals (errors)

# # Residual plot: how accurate is our regression model?
# plt.scatter(model.fittedvalues, model.resid, alpha=0.4)
# plt.axhline(0, color="red", ls="--")
# plt.xlabel("Prediction")
# plt.ylabel("Residual")
# plt.title("Residual Plot")
# plt.show()

KeyError: ['y', 'x1', 'x2']

---
# Task 1: Promo2 revisited
Dataset: Rossmann Store Sales

## Task 1.1 – Recap: Record Conclusions and Variables

Note briefly:
- **Conclusion from last week** regarding Promo2 (one sentence)
- **Columns used** from the Rossmann dataset
- **Feature-engineered columns** additionally created

**Conclusion (last week):** Based on the line plot, Promo2 appeared to have a small positive effect on sales rather than a fading "hype" effect, since average sales of Promo2 stores stayed broadly stable across the 36 months following campaign launch.

**Columns used:** `Promo2`, `Promo2SinceWeek`, `Promo2SinceYear`, `Date`, `Sales`, `Open` 

**Feature-engineered columns:** `Promo2Start` (campaign start date, derived from `Promo2SinceWeek` + `Promo2SinceYear`) and `months_since_promo2_start` (elapsed months between `Promo2Start` and `Date`).

## Task 1.2 – Formulate and Test Hypotheses

**Hypotheses:**
- H0: β_Promo2 = 0  → Promo2 has **no** effect on Sales
- H1: β_Promo2 ≠ 0  → Promo2 has **some** effect on Sales

**Business perspective:**
- **H0:** Running Promo2 makes no real difference to sales  
- **H1:** Promo2 systematically changes sales (up or down), making the campaign a real business lever worth evaluating.

In [ ]:
# load and merge the Rossmann data
train = pd.read_csv('data-rossmann/train.csv', low_memory=False)
train['Date'] = pd.to_datetime(train['Date'], format='%Y-%m-%d')
store = pd.read_csv('data-rossmann/store.csv')
df = train.merge(store, on='Store', how='left')

# only open days with positive sales (closed days trivially have Sales = 0)
df_open = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()

# fit the naive model
m_naiv = smf.ols("Sales ~ Promo2", data=df_open).fit()
print(m_naiv.summary())

# extract key values
params = m_naiv.params
pvals  = m_naiv.pvalues
conf   = m_naiv.conf_int()
r2     = m_naiv.rsquared

print()
print(f"coef Promo2 : {params['Promo2']:.2f}")
print(f"p-value     : {pvals['Promo2']:.3g}")
print(f"95% CI      : [{conf.loc['Promo2', 0]:.2f}, {conf.loc['Promo2', 1]:.2f}]")
print(f"R-squared   : {r2:.4f}")

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.016
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                 1.397e+04
Date:                Sat, 06 Jun 2026   Prob (F-statistic):               0.00
Time:                        18:42:32   Log-Likelihood:            -7.9799e+06
No. Observations:              844338   AIC:                         1.596e+07
Df Residuals:                  844336   BIC:                         1.596e+07
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   7350.8184      4.732   1553.538      0.0

**Estimated coefficient of Promo2:**  -> 791.82 (Promo2 stores sell ~€792/day less on average)

**p-value interpretation:** ≈ 0 the difference is highly unlikely to be due to chance

**95% Confidence Interval:** [−804.96, −778.69] 

**R²:** 0.016 Promo2 alone explains only 1.6% of the variation in Sales

**Overall interpretation (for a manager):** *"Stores running the campaign sell about €792 less per day on average, and this isn't a coincidence . But we can't yet say whether the campaign causes this or whether those stores were simply different to begin with."*

**Reject H0?** Yes, p ≈ 0 and the CI excludes zero, so we reject H0 in favour of H1.

## Task 1.3 – Comparison with Last Week

Does the direction of the effect (positive / negative) agree with last week's conclusion?

> **Hint:** If the sign flipped, think about what this simple model might be missing  
> (e.g. store size, location → Omitted Variable Bias).

**Direction agreement:** No, last week suggested a positive effect, but the model shows a significant **negative** coefficient (−€792/day).

**Possible explanation if it diverges:** Omitted Variable Bias, stores choose whether to run Promo2, so participating stores may differ systematically (size, location, store type) in ways that affect sales independently of the campaign. The model attributes that whole difference to `Promo2`.

## Task 1.4 – Does the Promo Effect Fade Over Time?

Filter to stores that actually participate in Promo2, then regress Sales against
`months_since_promo2_start` (the feature engineered last week).

In [11]:
# filter for stores that actually participate in Promo2 and have a valid start date
df_promo2 = df_open[(df_open['Promo2'] == 1) & df_open['Promo2SinceWeek'].notna()].copy()

# build months_since_promo2_start (campaign start = Monday of Promo2SinceWeek/Promo2SinceYear)
df_promo2['Promo2Start'] = pd.to_datetime(
    df_promo2['Promo2SinceYear'].astype(int).astype(str)
    + '-W'
    + df_promo2['Promo2SinceWeek'].astype(int).astype(str).str.zfill(2)
    + '-1',
    format='%G-W%V-%u'
)
df_promo2['months_since_promo2_start'] = (
    (df_promo2['Date'] - df_promo2['Promo2Start']).dt.days / 30.44
).astype(int)

# keep only rows after campaign launch, within a 36-month window (matches the data's 2.5-year span)
df_promo2 = df_promo2[
    (df_promo2['months_since_promo2_start'] >= 0)
    & (df_promo2['months_since_promo2_start'] <= 36)
]

# fit the time-trend model
m_time = smf.ols("Sales ~ months_since_promo2_start", data=df_promo2).fit()
print(m_time.summary())

# extract key values
params_t = m_time.params
pvals_t  = m_time.pvalues
conf_t   = m_time.conf_int()
r2_t     = m_time.rsquared

print()
print(f"coef months_since_promo2_start : {params_t['months_since_promo2_start']:.2f}")
print(f"p-value                        : {pvals_t['months_since_promo2_start']:.3g}")
print(f"95% CI                         : [{conf_t.loc['months_since_promo2_start', 0]:.2f}, {conf_t.loc['months_since_promo2_start', 1]:.2f}]")
print(f"R-squared                      : {r2_t:.4f}")

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     405.3
Date:                Sat, 06 Jun 2026   Prob (F-statistic):           4.66e-90
Time:                        21:13:32   Log-Likelihood:            -2.1676e+06
No. Observations:              233063   AIC:                         4.335e+06
Df Residuals:                  233061   BIC:                         4.335e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

**Sign of coefficient (months_since_promo2_start):** Positive (+10.45)

**Business interpretation:** The effect does not fade -> sales stay stable or even rise slightly the longer the campaign runs.

**Certainty (p-value, CI):** p ≈ 4.7 × 10⁻⁹⁰ and 95% CI [9.43, 11.46] -> both confirm the positive trend is real, though R² = 0.0017 means the practical size of this trend is small.

## Task 1.5 – Reformulate the Conclusion

Write **one sentence** that answers the business question based on your inference.
Compare with your answer from Task 1.1.

**Conclusion:** Sales rise slightly but steadily the longer a store runs Promo2 (+€10.45/month, no fade-out), yet Promo2 stores still sell about €792/day less than non-Promo2 stores on average, so compared to last week's "Promo2 boosts sales" takeaway, the honest answer is that the campaign's effect persists over time but cannot be credited with raising sales overall (Omitted Variable Bias).

---
# Task 2: Robot Arm – Temperature and Positioning Accuracy

**Dataset:** `robot_arm_degradation.csv`  
**Key columns:**
- `temp_mean_J1` – mean temperature of joint J1
- `diff_pos_J1_std` – std of positional deviation in J1 (higher = less accurate)

**Question:** Does increasing joint temperature reduce positioning accuracy?

## Task 2.1 – Fit the Regression Model

In [ ]:
# TODO: Load the robot arm dataset (adjust path as needed)
# df_robot = pd.read_csv("robot_arm_degradation.csv")

# Quick inspection
# print(df_robot[['temp_mean_J1', 'diff_pos_J1_std']].describe())

# Drop NaNs in the relevant columns before fitting
# df_robot_clean = df_robot.dropna(subset=['temp_mean_J1', 'diff_pos_J1_std'])

# Fit the model
# m_robot = smf.ols("diff_pos_J1_std ~ temp_mean_J1", data=df_robot_clean).fit()
# print(m_robot.summary())

# Extract key values
# params_r = m_robot.params
# pvals_r  = m_robot.pvalues
# conf_r   = m_robot.conf_int()
# r2_r     = m_robot.rsquared

## Task 2.2 – Interpret the Inference

**Hypotheses:**
- H0: β_temp_mean_J1 = 0  → Temperature has no effect on positioning accuracy
- H1: β_temp_mean_J1 ≠ 0  → Temperature affects positioning accuracy

**Slope (accuracy loss per °C):** *TODO – include units (mm/°C)*

**95% Confidence Interval:** *TODO*

**p-value – sufficient evidence to reject H0?:** *TODO*

**R² – how much variance does temperature alone explain?:** *TODO*

## Task 2.3 – Residual Plot

In [ ]:
# Residual plot for the robot arm model
# plt.scatter(m_robot.fittedvalues, m_robot.resid, alpha=0.4)
# plt.axhline(0, color="red", ls="--")
# plt.xlabel("Fitted values (predicted diff_pos_J1_std)")
# plt.ylabel("Residuals")
# plt.title("Residual Plot – Robot Arm (J1 Temperature)")
# plt.tight_layout()
# plt.show()

**Residual pattern observed:** *TODO – cloud, funnel, curve, or clusters?*

**Are the basic OLS assumptions met?:** *TODO*

## Task 2.4 – Statistical vs. Practical Significance

Even with p < 0.001, the effect might be too small to matter in practice.

In [ ]:
# TODO: Calculate the total predicted accuracy loss across the observed temperature range
# temp_range = df_robot_clean['temp_mean_J1'].max() - df_robot_clean['temp_mean_J1'].min()
# slope      = m_robot.params['temp_mean_J1']
# total_loss = slope * temp_range
# print(f"Temperature range: {temp_range:.2f} °C")
# print(f"Total predicted accuracy loss: {total_loss:.6f} mm")

# TODO: Compare to mean positional deviation
# mean_dev = df_robot_clean['diff_pos_J1_std'].mean()
# print(f"Mean positional deviation (diff_pos_J1_std): {mean_dev:.6f} mm")
# print(f"Ratio loss / mean deviation: {total_loss / mean_dev:.4f}")

**Total accuracy loss over full temperature range:** *TODO mm*

**Mean positional deviation (baseline):** *TODO mm*

**Is the effect practically relevant for an engineer?:** *TODO*

**Recommendation to the design team:** *TODO*

---
# Task 3: Reflection

## Task 3 – Reflection

Compare both datasets and models across the following dimensions.

**Note:** `robot_arm_degradation.csv` is missing, so `m_robot` doesn't exist yet — the points below state the general principle from the Rossmann models; the direct comparison still needs Task 2.1.

**Narrower CIs / smaller p-values:** Larger samples shrink standard errors (∝ 1/√n), giving narrower CIs and smaller p-values for the same true effect. The Rossmann models use huge samples (n ≈ 233k–844k), hence the tight CI [9.43, 11.46] and p ≈ 4.7 × 10⁻⁹⁰.

**Role of sample size:** n controls the *precision* of an estimate — larger n means a tiny, practically irrelevant effect (+€10/month, R² = 0.0017) can still look "highly significant". Statistical significance shows an effect is *probably real*, not that it's *large enough to matter*.</cell id="cell-t3-placeholder">